In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.subscription_events")
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")
T_vouchTrans = spark.table("cpx_dataanalytics_gold.marketing.tbltransvoucher")
T_vouchDesc = spark.table("cpx_dataanalytics_gold.marketing.tblvouchertype")

## SignUps

In [0]:
df_cte_a = (
    T_subscriber
    .filter(F.col("renewal_status") == "Active_Annual_Upgrade")
    .groupBy("cde_id")
    .agg(
        F.min(
            F.to_date(
                F.col("snapshot_date_id").cast("string"),
                "yyyyMMdd"
            )
        ).alias("date_upgraded")
    )
)

# Final aggregation
df_result = (
    df_cte_a
    .filter(
        F.col("date_upgraded").between("2025-07-01", "2025-10-31")
    )
    .withColumn("mm", F.month("date_upgraded"))
    .groupBy("mm")
    .agg(
        F.countDistinct("cde_id").alias("no_of_upgrades")
    )
    .orderBy("mm")
)

df_result.display()

In [0]:
# Equivalent of CTE_A
df_upgrades = (
    T_subscriber
    .filter(F.col("renewal_status") == "Active_Annnual_Upgrade")
    .groupBy("cde_id")
    .agg(
        F.min(
            F.to_date(
                F.col("snapshot_date_id").cast("string"),
                "yyyyMMdd"
            )
        ).alias("date_upgraded")
    )
)

# Final aggregation
df_result = (
    df_upgrades
    .filter(
        F.col("date_upgraded").between("2025-07-01", "2025-10-31")
    )
    .groupBy(F.month("date_upgraded").alias("mm"))
    .agg(
        F.countDistinct("cde_id").alias("no_of_upgrades")
    )
    .orderBy("mm")
)

display(df_result)


In [0]:
%sql
With cte_a as
(
  select cde_id
      ,min(to_date(cast(snapshot_date_id as string),'yyyMMdd')) as date_upgraded
from cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot
where renewal_status = 'Active_Annnual_Upgrade'
group by cde_id
)
select month(date_upgraded) as mm
      ,count(distinct cde_id) as no_of_upgrades
from cte_a
where date_upgraded between '2025-07-01' and '2025-10-31'
group by month(date_upgraded)
order by month(date_upgraded)

In [0]:
df_upgraded_new= (
    T_subscriber.filter((F.col("snapshot_date_id") == '20251101') & (F.col("renewal_status").like("%Active%")))
    .join(T_events, "cc_user_profile_id", how="left")
    .filter(
        (F.col("reporting_datetime").between("2025-10-01", "2025-10-31")) &
        (F.col("changed") == True) &
        (F.col("changed_from").like("%monthly%")) &
        (F.col("changed_to").like("%annual%"))
    )
)

df_upgraded_new.distinct().count()


In [0]:
df_members_renewed_new = (
    T_subscriber.filter(F.col("snapshot_date_id") == 20251101)
    .join(T_events, "cc_user_profile_id", how="inner")
    .filter(
        (F.col("reporting_datetime").between("2025-10-01", "2025-10-31")) &
        (F.col("event_type") == "subscription_renewed") &
        (F.col("status") == "active") &
        (T_events.plan_id.like("%annual%")) &
        (F.col("total") > 90)
    )
)

In [0]:
df_members_renewed_new.count()

In [0]:
df_details = (
    T_subscriber.filter(F.col("snapshot_date_id") == 20251101)
    .filter(F.to_date(T_subscriber.activated_at_datetime).between("2025-10-01", "2025-10-31"))
    .filter(F.col("plan_id") == "annual1")
    .select('cde_id',T_subscriber.activated_at_datetime)
    .distinct()
)

df_first_activation_new = (
    T_subscriber
    .groupBy("cde_id")
    .agg(
        F.min(F.to_date(T_subscriber.activated_at_datetime))
    )
    .withColumnRenamed("min(to_date(activated_at_datetime))","first_activation")
)


df_details.count()

In [0]:
df_details = (df_details
              .join(df_first_activation_new, on="cde_id", how="inner")
              )

df_returning = (
    df_details
    .filter(~F.col("first_activation").between("2025-10-01", "2025-10-31"))
)

df_new_signups = (
    df_details
    .filter(F.col("first_activation").between("2025-10-01", "2025-10-31"))
)


In [0]:
df_returning = (
    df_details
    .filter(F.col("first_activation").between("2025-10-01", "2025-10-31"))
)

df_returning.count()

In [0]:
df_merged = (
    df_upgraded_new.select("cde_id")
    .union(df_members_renewed_new.select("cde_id"))
    .union(df_details.select("cde_id"))
    .withColumnRenamed("cde_id","CDE_ID")
    .distinct()
)

In [0]:
df_all_trans = (
    df_merged
    .join(T_trans, "CDE_ID", how="left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_items.select("itemskey", "Item_Description"), T_trans.ItemSkey == T_items.itemskey, "left")
    .filter(F.col("datefull").between("2025-01-01", "2025-12-20"))
)

In [0]:
df_p12 = (
    df_all_trans
    .filter(
        F.col("datefull").between("2025-01-01", "2025-07-20")
    )
)

df_visits_p12 = (
    df_p12
    .groupBy("CDE_ID", "datefull", "LocationId")
    .agg(
        # Did popcorn occur in this visit?
        F.max(
            F.when(
                F.col("Item_Description").rlike("(?i)popcorn"), 1
            ).otherwise(0)
        ).alias("Popcorn_Visit_Flag")
    )
)

df_guest_p12_summary = (
    df_visits_p12
    .groupBy("CDE_ID")
    .agg(
        F.count("*").alias("Total_Visits_P12"),
        F.sum("Popcorn_Visit_Flag").alias("Popcorn_Visit_Count_P12")
    )
    .withColumn(
        "Popcorn_Incidence_Rate",
        F.col("Popcorn_Visit_Count_P12") / F.col("Total_Visits_P12")
    )
)


In [0]:
df_guest_p12_summary.display()

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta

# -------------------------------------------------------
# 1. USER-CONTROLLED PARAMETERS
# -------------------------------------------------------

# Snapshot (YYYYMMDD as int, same as subscriber snapshot_date_id)
SNAPSHOT_DATE_ID = "20250801"

# Convert snapshot to a proper date
SNAPSHOT_DATE = F.to_date(F.lit(str(SNAPSHOT_DATE_ID)), "yyyyMMdd")

# Event / activation analysis windows (editable)
EVENT_START_DATE = "2025-07-01"
EVENT_END_DATE   = "2025-07-31"

ACTIVATION_START_DATE = "2025-07-01"
ACTIVATION_END_DATE   = "2025-07-31"

# -------------------------------------------------------
# 2. DYNAMIC P12 WINDOW (12 months prior to snapshot)
# -------------------------------------------------------

P12_END_DATE   = F.date_sub(SNAPSHOT_DATE, 1)
P12_START_DATE = F.add_months(P12_END_DATE, -12)

# -------------------------------------------------------
# 3. UPGRADED MEMBERS (Monthly → Annual)
# -------------------------------------------------------

df_upgraded_new = (
    T_subscriber
    .filter(F.col("snapshot_date_id") == SNAPSHOT_DATE_ID)
    .join(T_events, "cc_user_profile_id", "inner")
    .filter(
        (F.col("reporting_datetime").between(EVENT_START_DATE, EVENT_END_DATE)) &
        (F.col("changed") == True) &
        (F.col("changed_from").like("%monthly%")) &
        (F.col("changed_to").like("%annual%"))
    )
    .select("cde_id")
)

# -------------------------------------------------------
# 4. RENEWED ANNUAL MEMBERS
# -------------------------------------------------------

# df_members_renewed_new = (
#     T_subscriber
#     .filter(F.col("snapshot_date_id") == SNAPSHOT_DATE_ID)
#     .join(T_events, "cc_user_profile_id", "inner")
#     .filter(
#         (F.col("reporting_datetime").between(EVENT_START_DATE, EVENT_END_DATE)) &
#         (F.col("event_type") == "subscription_renewed") &
#         (F.col("status") == "active") &
#         (T_events.plan_id.like("%annual%")) &
#         (F.col("total") > 90)
#     )
#     .select("cde_id")
# )


# -------------------------------------------------------
# 5. NEW ANNUAL ACTIVATIONS
# -------------------------------------------------------
df_details = (
    T_subscriber.filter(F.col("snapshot_date_id") == SNAPSHOT_DATE_ID)
    .filter(F.to_date(T_subscriber.activated_at_datetime).between(ACTIVATION_START_DATE, ACTIVATION_END_DATE))
    .filter((F.col("plan_id") == "annual1") & (F.col("renewal_status").like("%Active%")))
    .select('cde_id')
    .distinct()
)




# -------------------------------------------------------
# 6. MERGED MEMBER COHORT
# -------------------------------------------------------

df_merged = (
    df_upgraded_new
    # .union(df_members_renewed_new)
    .union(df_details)
    .withColumnRenamed("cde_id", "CDE_ID")
    .distinct()
)
display("Total Member Count")
display(df_details.count())

# -------------------------------------------------------
# 7. TRANSACTIONS WITHIN P12 WINDOW
# -------------------------------------------------------

df_all_trans = (
    df_merged
    .join(T_trans, "CDE_ID", "left")
    .join(
        T_date.select("dateid", "datefull"),
        T_trans.VisitDateId == T_date.dateid,
        "left"
    )
    .join(
        T_items.select("itemskey", "Item_Description"),
        T_trans.ItemSkey == T_items.itemskey,
        "left"
    )
    .filter(F.col("datefull").between(P12_START_DATE, P12_END_DATE))
)

# -------------------------------------------------------
# 8. VISIT-LEVEL AGGREGATION (P12)
# -------------------------------------------------------

df_visits_p12 = (
    df_all_trans
    .groupBy("CDE_ID", "datefull", "LocationId")
    .agg(
        F.max(
            F.when(
                F.col("Item_Description").rlike("(?i)popcorn"), 1
            ).otherwise(0)
        ).alias("Popcorn_Visit_Flag")
    )
)

# -------------------------------------------------------
# 9. GUEST-LEVEL P12 SUMMARY
# -------------------------------------------------------

df_guest_p12_summary = (
    df_visits_p12
    .groupBy("CDE_ID")
    .agg(
        F.count("*").alias("Total_Visits_P12"),
        F.sum("Popcorn_Visit_Flag").alias("Popcorn_Visit_Count_P12")
    )
    .withColumn(
        "Popcorn_Incidence_Rate",
        F.col("Popcorn_Visit_Count_P12") / F.col("Total_Visits_P12")
    )
)


In [0]:
from pyspark.sql import functions as F

# -------------------------------------------------------
# 1. Guest-level total visit count (P12)
# -------------------------------------------------------

df_guest_visits = (
    df_visits_p12
    .groupBy("CDE_ID")
    .agg(
        F.count("*").alias("Total_Visits_P12")
    )
)

# -------------------------------------------------------
# 2. Assign visit-frequency buckets
# -------------------------------------------------------

df_guest_buckets = (
    df_guest_visits
    .withColumn(
        "Visit_Bucket",
        F.when(F.col("Total_Visits_P12")== 0, "0")
         .when(F.col("Total_Visits_P12") == 1, "1")
         .when(F.col("Total_Visits_P12").between(2, 3), "2-3")
         .when(F.col("Total_Visits_P12").between(4, 6), "4-6")
         .when(F.col("Total_Visits_P12").between(7, 12), "7-12")
         .otherwise("13+")
    )
)

# -------------------------------------------------------
# 3. Distribution: distinct guests per bucket
# -------------------------------------------------------

df_visit_distribution = (
    df_guest_buckets
    .groupBy("Visit_Bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("Distinct_Guests")
    )
)

# -------------------------------------------------------
# 4. Overall average visits (P12)
# -------------------------------------------------------

df_avg_visits = (
    df_guest_visits
    .agg(
        F.avg("Total_Visits_P12").alias("Avg_Visits_P12")
    )
)


In [0]:
df_guest_visits.display()

In [0]:
df_visit_distribution.display()

In [0]:
df_avg_visits.display()

In [0]:
df_all_trans_Combo = (
    T_subscriber.filter((F.col("snapshot_date_id") == 20251001) & (F.col("plan_id").like("annual1")) & (F.col("Renewal_status").like("annnual%")))
    .join(T_trans, "CDE_ID", how="left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_items.select("itemskey", "Item_ID","Item_Description"), T_trans.ItemSkey == T_items.itemskey, "left")
    .filter(F.col("datefull").between("2025-10-01", "2025-10-31"))
    .filter(F.col("Item_Description").like("(i?)Reese%"))
    .groupBy("CDE_ID", "datefull", "Item_Description",)
    .agg(
        F.sum(F.col("Quantity"))
    )
    .distinct()
)


df_all_trans_Combo.display()

In [0]:
df_voucher_trans = (
    T_vouchTrans.alias("vt")
    .join(
        T_vouchDesc.alias("vd"),
        F.col("vt.voucher_type_id") == F.col("vd.voucher_type_id"),
        how="left"
    )
    .filter(F.col("n_voucher_code").like("7132%"))
)

display(df_voucher_trans.count())

In [0]:
from pyspark.sql import functions as F

V1 = df_voucher_trans.alias("V1")
V3 = T_trans.alias("V3")

df_voucher_trans_joined = (
    V1
    .withColumn(
        "session_date_id",
        F.date_format(F.col("session_dtm_realshow"), "yyyyMMdd")
    )
    .join(
        V3.select("CDE_ID", "LocationId", "Transaction_Number", "VisitDateId"),
        (V1.l_location_id == V3.LocationId) &
        (V1.trans_c_lgn_number == V3.Transaction_Number) &
        (F.col("session_date_id") == V3.VisitDateId),
        how="inner"
    )
)

display(df_voucher_trans_joined.select(F.col("transaction_voucher_id")).distinct().count())

In [0]:
df_voucher_trans_joined = (
    V1
    .withColumn(
        "session_date_id",
        F.date_format(F.col("session_dtm_realshow"), "yyyyMMdd")
    )
    .join(
        V3.select(
            "CDE_ID",
            "LocationId",
            "Transaction_Number",
            "VisitDateId"
        ),
        (V1.l_location_id == V3.LocationId) &
        (V1.trans_c_lgn_number == V3.Transaction_Number) &
        (F.col("session_date_id") == V3.VisitDateId),
        how="left"
    )
    .withColumn(
        "Transaction_Matched_Flag",
        F.when(F.col("Transaction_Number").isNotNull(), 1).otherwise(0)
    )
    .select("CDE_ID")
    .distinct()
)


In [0]:
df_popcorn_incidence = (
    df_voucher_trans_joined
    .join(T_trans, "CDE_ID", how="left")
    .join(T_subscriber.filter(F.col("snapshot_date_id") == '20251001'), "CDE_ID", how="left")
    .join(
        T_items.select("itemskey", "Item_Description"),
        T_trans.ItemSkey == T_items.itemskey,
        "left"
    )
    .withColumn(
        "Visit_Date",
        F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd")
    )
    .filter(F.col("Visit_Date").between("2024-11-01", "2025-11-30"))
)

df_popcorn_visits = (
    df_popcorn_incidence
    .groupBy("CDE_ID", "VisitDateId", "LocationId", "plan_id")
    .agg(
        F.max(
            F.when(
                F.col("Item_Description").rlike("(?i)nachos"), 1
            ).otherwise(0)
        ).alias("Nachos_Visit_Flag")
    )
)

df_popcorn_incidence_final = (
    df_popcorn_visits
    .groupBy("CDE_ID", "plan_id")
    .agg(
        F.count("*").alias("Total_Visits"),
        F.sum("Nachos_Visit_Flag").alias("Nachos_Visits")
    )
    .withColumn(
        "Nachos_Incidence",
        F.col("Nachos_Visits") / F.col("Total_Visits")
    )
)


In [0]:
df_popcorn_incidence_final.display()

In [0]:
df_active_annual = (
    T_subscriber
    .filter(F.col("snapshot_date_id") == 20251220)
    .filter(F.col("renewal_status").like("%Active%"))
    .filter(F.col("plan_id").like("%annual%"))
    .select("cde_id")
    .distinct()
)

df_upgraded_annual = (
    T_subscriber
    .filter(F.col("renewal_status") == "Active_Annnual_Upgrade")
    .groupBy("cde_id")
    .agg(
        F.min(
            F.to_date(
                F.col("snapshot_date_id").cast("string"),
                "yyyyMMdd"
            )
        ).alias("date_upgraded")
    )
    .select("cde_id")
    .distinct()
)

df_eligible_members = (
    df_active_annual
    .union(df_upgraded_annual)
    .distinct()
)

df_visits_p12 = (
    T_trans
    .join(T_date.select("dateid", "datefull"),
          T_trans.VisitDateId == T_date.dateid,
          "left")
    .filter(
        F.col("datefull").between(
            F.add_months(F.current_date(), -12),
            F.current_date()
        )
    )
    .groupBy("cde_id", "datefull")
    .agg(F.lit(1).alias("visit_flag"))
    .groupBy("cde_id")
    .agg(F.count("*").alias("total_visits_p12"))
)


In [0]:
snapshot_date = F.to_date(F.lit("20251101"), "yyyyMMdd")
start_date = F.add_months(snapshot_date, -12)
end_date = snapshot_date

T_master_trans = (T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_items, T_trans.ItemSkey == T_items.itemskey, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre")
        & (F.col("STOREFLAG") == "No")
        & (F.col("RecRoomFLAG") == "No")
        & (T_trans.CDE_ID.isNotNull())
        & (F.col("datefull").between(start_date, end_date))
        & ((F.col("TicketFLAG") == "Yes") | (F.col("ConcessionFLAG") == "Yes"))
    )
    .withColumnRenamed("datefull", "VisitDate")
)


In [0]:
# Define upgrade window (inclusive)
upgrade_start_date = "2025-10-01"
upgrade_end_date   = "2025-10-31"

snapshot_date_id = "20251101"

# CTE_A logic in PySpark
df_upgraded_members = (
    T_subscriber
    .filter(F.col("renewal_status") == "Active_Annnual_Upgrade")
    .groupBy("cde_id")
    .agg(
        F.min(
            F.to_date(
                F.col("snapshot_date_id").cast("string"),
                "yyyyMMdd"
            )
        ).alias("date_upgraded")
    )
    .filter(F.col("date_upgraded").between(upgrade_start_date, upgrade_end_date))
    .select("cde_id")
    .distinct()
)


df_active_annual_snapshot = (
    T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date_id)
    .filter(F.to_date(T_subscriber.activated_at_datetime).between(upgrade_start_date, upgrade_end_date))
    .filter((F.col("plan_id") == "annual1") & (F.col("renewal_status").like("%Active%")))
    .select('cde_id',T_subscriber.activated_at_datetime)
    .distinct()
)


df_first_activation_new = (
    T_subscriber
    .groupBy("cde_id")
    .agg(
        F.min(F.to_date(T_subscriber.activated_at_datetime))
    )
    .withColumnRenamed("min(to_date(activated_at_datetime))","first_activation")
)

df_details = (df_active_annual_snapshot
              .join(df_first_activation_new, on="cde_id", how="inner")
              )

df_returning = (
    df_details
    .filter(~F.col("first_activation").between(upgrade_start_date, upgrade_end_date))
)

df_new_signups = (
    df_details
    .filter(F.col("first_activation").between(upgrade_start_date, upgrade_end_date))
)

display("Upgraded_members = ", df_upgraded_members.count())
display("\nReturning Members = ", df_returning.count())
display("\nNew Members = ", df_new_signups.count())

# df_active_annual_snapshot = (
#     T_subscriber
#     .filter(F.col("snapshot_date_id") == snapshot_date_id)
#     .filter(F.col("renewal_status").like("%Active%"))
#     .filter(F.col("plan_id").like("%annual1%"))
#     .select("cde_id")
#     .distinct()
# )

df_eligible_members = (
    df_upgraded_members
    .union(df_details.select("cde_id"))
    .distinct()
)

display("\nTotal Members = ", df_eligible_members.count())

df_all_member_trans = df_eligible_members.join(T_master_trans, "CDE_ID", "inner")

In [0]:
df_upgraded_flag = (
    df_upgraded_members
    .withColumn("is_upgraded", F.lit(1))
)

df_returning_flag = (
    df_returning
    .select("cde_id")
    .withColumn("is_returning", F.lit(1))
)

df_new_flag = (
    df_new_signups
    .select("cde_id")
    .withColumn("is_new", F.lit(1))
)

df_classification = (
    df_upgraded_flag
    .join(df_returning_flag, "cde_id", "full")
    .join(df_new_flag, "cde_id", "full")
    .fillna(0)
)

df_final_segments = (
    df_classification
    .withColumn(
        "member_type",
        F.when(F.col("is_upgraded") == 1, "Upgraded")
         .when(F.col("is_new") == 1, "New")
         .when(F.col("is_returning") == 1, "Returning")
         .otherwise("Unclassified")
    )
)

df_distribution = (
    df_final_segments
    .groupBy("member_type")
    .agg(F.countDistinct("cde_id").alias("member_count"))
    .orderBy("member_type")
)

display(df_distribution)


In [0]:
df_visits = (
    df_all_member_trans
    .select("CDE_ID", "VisitDate")
    .dropDuplicates()
)

df_visit_counts = (
    df_visits
    .groupBy("CDE_ID")
    .agg(
        F.count("*").alias("total_visits_p12")
    )
)

df_member_visits = (
    df_eligible_members
    .join(df_visit_counts, "CDE_ID", "left")
    .fillna(0, subset=["total_visits_p12"])
)

df_visit_history = (
    df_member_visits
    .withColumn(
        "visit_bucket",
        F.when(F.col("total_visits_p12") == 0, "0")
         .when(F.col("total_visits_p12") == 1, "1")
         .when(F.col("total_visits_p12").between(2, 3), "2–3")
         .when(F.col("total_visits_p12").between(4, 6), "4–6")
         .when(F.col("total_visits_p12").between(7, 12), "7–12")
         .otherwise("13+")
    )
)

df_visit_category_counts = (
    df_visit_history
    .groupBy("visit_bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("no_of_people")
    )
    .orderBy("visit_bucket")
)


In [0]:
df_visit_category_counts.display()